In [ ]:
# pip install numpy matplotlib scipy scikit-learn

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde


# ------------------------------------------------------------
# 1. Example high-dimensional data
# ------------------------------------------------------------
X, labels = make_blobs(
    n_samples=3000,
    n_features=8,
    centers=5,
    cluster_std=[0.7, 1.0, 1.4, 0.9, 1.2],
    random_state=42,
)

X = StandardScaler().fit_transform(X)


# ------------------------------------------------------------
# 2. PCA to 2D
# ------------------------------------------------------------
pca = PCA(n_components=2)
Z = pca.fit_transform(X)

pc1 = Z[:, 0]
pc2 = Z[:, 1]

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())


# ------------------------------------------------------------
# 3. KDE on the 2D PCA projection
# ------------------------------------------------------------
kde = gaussian_kde(
    np.vstack([pc1, pc2]),
    bw_method=0.25,   # lower = sharper, higher = smoother
)


# ------------------------------------------------------------
# 4. Evaluate KDE on a grid
# ------------------------------------------------------------
pad = 1.0

xmin, xmax = pc1.min() - pad, pc1.max() + pad
ymin, ymax = pc2.min() - pad, pc2.max() + pad

xx, yy = np.meshgrid(
    np.linspace(xmin, xmax, 300),
    np.linspace(ymin, ymax, 300),
)

grid = np.vstack([xx.ravel(), yy.ravel()])
density = kde(grid).reshape(xx.shape)


# ------------------------------------------------------------
# 5. Plot filled density + contour lines
# ------------------------------------------------------------
plt.figure(figsize=(9, 7))

filled = plt.contourf(
    xx,
    yy,
    density,
    levels=30,
    cmap="viridis",
)

lines = plt.contour(
    xx,
    yy,
    density,
    levels=12,
    colors="white",
    linewidths=0.8,
)

plt.scatter(
    pc1,
    pc2,
    s=4,
    c="black",
    alpha=0.18,
)

plt.clabel(lines, inline=True, fontsize=8)

plt.colorbar(filled, label="KDE density on PCA projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("2D PCA density estimate with contour lines")
plt.tight_layout()
plt.show()